In [ ]:
import requests
import json

# Define the URL for the API
url = "http://localhost:8000/v1/chat/completions"

# Set up the headers
headers = {
    "Content-Type": "application/json"
}

# Set up the data for the request
data = {
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [
        {
            "role": "user",
            "content": "How to stop procastinating"
        }
    ]
}

# Send the POST request
response = requests.post(url, headers=headers, data=json.dumps(data))

# Check if the request was successful
if response.status_code == 200:
    # Parse the response JSON
    response_json = response.json()
    # Print the model's reply
    print("Model's response:", response_json['choices'][0]['message']['content'])
else:
    print(f"Failed to get a response. Status code: {response.status_code}")


Creation of database

In [2]:
import sqlite3
import os
from faker import Faker
import random

fake = Faker()

def generate_row():
    """Generate a row of data for the large database."""
    return (
        fake.uuid4(),
        fake.name(),
        fake.email(),
        fake.address(),
        fake.phone_number(),
        fake.job(),
        fake.company(),
        fake.date_of_birth().strftime('%Y-%m-%d'),
        fake.ssn(),
        fake.credit_card_number(),
        fake.credit_card_expire(),
        fake.credit_card_provider(),
        fake.country(),            # bank_country
        fake.currency_code(),
        round(random.uniform(100, 10000), 2),
        fake.date_time_this_decade().strftime('%Y-%m-%d %H:%M:%S')
    )

def create_large_database(db_name, target_size_bytes):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    # Create the full schema with primary key
    cur.execute('''
        CREATE TABLE IF NOT EXISTS customers (
            id TEXT PRIMARY KEY,
            name TEXT,
            email TEXT,
            address TEXT,
            phone_number TEXT,
            job TEXT,
            company TEXT,
            date_of_birth TEXT,
            ssn TEXT,
            credit_card_number TEXT,
            credit_card_expire TEXT,
            credit_card_provider TEXT,
            bank_country TEXT,
            currency_code TEXT,
            amount REAL,
            transaction_date TEXT
        )
    ''')
    conn.commit()

    count = 0
    while os.path.getsize(db_name) < target_size_bytes:
        row = generate_row()
        cur.execute('''
            INSERT INTO customers (
                id, name, email, address, phone_number, job, company, date_of_birth, ssn,
                credit_card_number, credit_card_expire, credit_card_provider, bank_country,
                currency_code, amount, transaction_date
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', row)
        count += 1
        if count % 1000 == 0:
            conn.commit()
            print(f"{db_name}: Inserted {count} rows; size: {os.path.getsize(db_name)} bytes")

    conn.commit()
    print(f"{db_name} created with {count} rows. Final size: {os.path.getsize(db_name)} bytes")
    conn.close()



# Target sizes
target_sizes = {
    'large_database.db': 1073741824,     # 1 GB
    '10mb_sample.db': 10485760,
    '20mb_sample.db': 20971520,
    '5mb_sample.db': 5242880
}

# Create large database with full schema
print(f"\nCreating large_database.db...")
create_large_database('large_database.db', target_sizes['large_database.db'])

# Create sample databases with only primary keys
sample_dbs = ['10mb_sample.db', '20mb_sample.db', '5mb_sample.db']
for db in sample_dbs:
    print(f"\nCreating {db} with only primary keys...")
    

KeyboardInterrupt: 

Creating sampled databases IDS

In [7]:
import sqlite3
import os
import random

def get_average_row_size(db_path, table_name='customers'):
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Database file {db_path} not found.")
    db_size = os.path.getsize(db_path)
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {table_name}")
    row_count = cur.fetchone()[0]
    conn.close()
    if row_count == 0:
        raise ValueError("No rows in the table.")
    return db_size / row_count

def sample_ids_only(large_db_path, sample_db_map, avg_row_size_bytes):
    conn = sqlite3.connect(large_db_path)
    cur = conn.cursor()
    cur.execute("SELECT id FROM customers")
    all_ids = [row[0] for row in cur.fetchall()]
    conn.close()
    random.shuffle(all_ids)

    for db_name, size_mb in sample_db_map.items():
        target_rows = int((size_mb * 1024 * 1024) / avg_row_size_bytes)
        print(f"Creating {db_name} with {target_rows} sampled IDs...")

        conn_sample = sqlite3.connect(db_name)
        cur_sample = conn_sample.cursor()
        cur_sample.execute("DROP TABLE IF EXISTS sample_ids")
        cur_sample.execute("CREATE TABLE sample_ids (id TEXT PRIMARY KEY)")
        for i in range(target_rows):
            cur_sample.execute("INSERT INTO sample_ids (id) VALUES (?)", (all_ids[i],))
            if i % 10000 == 0:
                conn_sample.commit()
        conn_sample.commit()
        conn_sample.close()

def enrich_sampled_dbs_with_full_rows(large_db_path, sample_db_map):
    for db_name in sample_db_map.keys():
        print(f"Enriching {db_name} with full customer rows from {large_db_path}...")

        # Load sampled IDs
        conn_sample = sqlite3.connect(db_name)
        cur_sample = conn_sample.cursor()
        cur_sample.execute("SELECT id FROM sample_ids")
        sampled_ids = [row[0] for row in cur_sample.fetchall()]

        # Prepare full customers table
        cur_sample.execute("DROP TABLE IF EXISTS customers")
        cur_sample.execute('''
            CREATE TABLE customers (
                id TEXT PRIMARY KEY,
                name TEXT,
                email TEXT,
                address TEXT,
                phone_number TEXT,
                job TEXT,
                company TEXT,
                date_of_birth TEXT,
                ssn TEXT,
                credit_card_number TEXT,
                credit_card_expire TEXT,
                credit_card_provider TEXT,
                bank_country TEXT,
                currency_code TEXT,
                amount REAL,
                transaction_date TEXT
            )
        ''')

        # Fetch rows from large DB
        conn_large = sqlite3.connect(large_db_path)
        cur_large = conn_large.cursor()
        full_rows = []
        batch_size = 500  # well below SQLite's 999-variable limit

        for i in range(0, len(sampled_ids), batch_size):
            batch = sampled_ids[i:i + batch_size]
            placeholder = ','.join(['?'] * len(batch))
            cur_large.execute(f"SELECT * FROM customers WHERE id IN ({placeholder})", batch)
            full_rows.extend(cur_large.fetchall())

        conn_large.close()

        # Insert into sample DB
        insert_query = '''
           
    INSERT INTO customers (
        id, name, email, address, phone_number, job, company, date_of_birth,
        ssn, credit_card_number, credit_card_expire, credit_card_provider,
        bank_country, currency_code, amount, transaction_date
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
'''

        for row in full_rows:
            cur_sample.execute(insert_query, row)
        conn_sample.commit()
        conn_sample.close()
        print(f"Finished enriching {db_name} with {len(full_rows)} rows.")

# Run both phases
sample_targets_mb = {
    '50mb_sample.db': 50,
    '100mb_sample.db': 100,
    '150mb_sample.db': 150,
    '200mb_sample.db': 200,
    '250mb_sample.db': 250
}

avg_row_size = get_average_row_size('large_database.db')
sample_ids_only('large_database.db', sample_targets_mb, avg_row_size)
enrich_sampled_dbs_with_full_rows('large_database.db', sample_targets_mb)


Creating 50mb_sample.db with 150908 sampled IDs...
Creating 100mb_sample.db with 301816 sampled IDs...
Creating 150mb_sample.db with 452724 sampled IDs...
Creating 200mb_sample.db with 603633 sampled IDs...
Creating 250mb_sample.db with 754541 sampled IDs...
Enriching 50mb_sample.db with full customer rows from large_database.db...


KeyboardInterrupt: 

Enrching sampled databases

In [8]:
import sqlite3
import os
from tqdm import tqdm

def enrich_sampled_dbs_with_full_rows(large_db_path, sample_db_paths, id_table="sample_ids"):
    insert_query = '''
        INSERT INTO customers (
            id, name, email, address, phone_number, job, company, date_of_birth,
            ssn, credit_card_number, credit_card_expire, credit_card_provider,
            bank_country, currency_code, amount, transaction_date
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    '''
    batch_size = 500  # Avoid "too many variables" error

    for sample_db in sample_db_paths:
        print(f"\n🔄 Enriching {sample_db} with full rows...")

        # Step 1: Read IDs from sample DB
        conn_sample = sqlite3.connect(sample_db)
        cur_sample = conn_sample.cursor()
        cur_sample.execute(f"SELECT id FROM {id_table}")
        sampled_ids = [row[0] for row in cur_sample.fetchall()]

        # Step 2: Create 'customers' table if not exists
        cur_sample.execute('DROP TABLE IF EXISTS customers')
        cur_sample.execute('''
            CREATE TABLE customers (
                id TEXT PRIMARY KEY,
                name TEXT,
                email TEXT,
                address TEXT,
                phone_number TEXT,
                job TEXT,
                company TEXT,
                date_of_birth TEXT,
                ssn TEXT,
                credit_card_number TEXT,
                credit_card_expire TEXT,
                credit_card_provider TEXT,
                bank_country TEXT,
                currency_code TEXT,
                amount REAL,
                transaction_date TEXT
            )
        ''')
        conn_sample.commit()

        # Step 3: Fetch full rows from large DB in batches
        conn_large = sqlite3.connect(large_db_path)
        cur_large = conn_large.cursor()

        all_rows = []
        for i in tqdm(range(0, len(sampled_ids), batch_size), desc=f"Fetching from {sample_db}"):
            batch = sampled_ids[i:i + batch_size]
            placeholders = ','.join(['?'] * len(batch))
            cur_large.execute(f"SELECT * FROM customers WHERE id IN ({placeholders})", batch)
            all_rows.extend(cur_large.fetchall())
        conn_large.close()

        # Step 4: Insert into sample DB in one transaction
        cur_sample.execute("BEGIN TRANSACTION")
        cur_sample.executemany(insert_query, all_rows)
        cur_sample.execute("COMMIT")
        conn_sample.close()

        print(f"✅ Finished {sample_db} with {len(all_rows)} full rows inserted.")

# ✅ Sample Usage
sample_db_paths = [
    '50mb_sample.db',
    '100mb_sample.db',
    '150mb_sample.db',
    '200mb_sample.db',
    '250mb_sample.db'
]

enrich_sampled_dbs_with_full_rows('large_database.db', sample_db_paths)



🔄 Enriching 50mb_sample.db with full rows...


Fetching from 50mb_sample.db: 100%|██████████| 302/302 [07:53<00:00,  1.57s/it]


✅ Finished 50mb_sample.db with 150908 full rows inserted.

🔄 Enriching 100mb_sample.db with full rows...


Fetching from 100mb_sample.db: 100%|██████████| 604/604 [11:05<00:00,  1.10s/it]


✅ Finished 100mb_sample.db with 301816 full rows inserted.

🔄 Enriching 150mb_sample.db with full rows...


...

✅ Finished 150mb_sample.db with 452724 full rows inserted.

🔄 Enriching 200mb_sample.db with full rows...


Fetching from 200mb_sample.db: 100%|██████████| 1208/1208 [14:15<00:00,  1.41it/s]


✅ Finished 200mb_sample.db with 603633 full rows inserted.

🔄 Enriching 250mb_sample.db with full rows...


Fetching from 250mb_sample.db: 100%|██████████| 1510/1510 [18:30<00:00,  1.36it/s]


✅ Finished 250mb_sample.db with 754541 full rows inserted.


In [1]:
import requests
import json
import sqlite3
import time
import csv

# Define the URL for the API
url = "http://localhost:8000/v1/chat/completions"

# Global variables to store metrics
execution_summary = []
time_detailsSM1 = []

# Set up the headers for API requests
headers = {
    "Content-Type": "application/json"
}

def ask_model(prompt):
    data = {
        "model": "defog/llama-3-sqlcoder-8b",
        "temperature": 0.0001,
        "messages": [{"role": "user", "content": prompt}]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    if response.status_code == 200:
        model_output = response.json()["choices"][0]["message"]["content"]
        print("\n🔍 Model Raw Output:\n", model_output)
        return model_output
    else:
        raise Exception(f"API request failed with status code {response.status_code}")

def natural_language_to_sql(user_command):
    prompt = f"""
    ### Task:
Convert the following natural language command into a correctly formatted SQLite SQL query.

### Important Notes:
1. Use only SQLite-compatible functions. SQLite does NOT support the following functions:
   - `EXTRACT`
   - `AGE`
   - `TO_DATE`
   - `DATE_PART`
   - `DATE_TRUNC`
2. For date calculations, use SQLite's `julianday` function.
3. Return ONLY the SQL query. NO explanations, NO markdown (`sql ... `), NO descriptions.
4. The output must be a direct SQL query, formatted correctly for SQLite.
5. Do NOT include anything else.

### Database Schema:
Table: customers
Columns:
  - id (TEXT, PRIMARY KEY)
  - name (TEXT)
  - email (TEXT)
  - address (TEXT)
  - phone_number (TEXT)
  - job (TEXT)
  - company (TEXT)
  - date_of_birth (TEXT)
  - ssn (TEXT)
  - credit_card_number (TEXT)
  - credit_card_expire (TEXT)
  - credit_card_provider (TEXT)
  - bank_country (TEXT)
  - currency_code (TEXT)
  - amount (REAL)
  - transaction_date (TEXT)

### User Input:
{user_command}

### SQL Output:
    """
    response = ask_model(prompt)
    if response.strip().upper().startswith("SELECT") and response.strip().endswith(";"):
        return response.strip()
    else:
        print("⚠️ Warning: Model response did not return a valid SQL query. Using fallback.")
        return "SELECT * FROM customers LIMIT 5;"

def execute_sql(database_path, sql_query, scaling_factor=1):
    try:
        conn = sqlite3.connect(database_path)
        cursor = conn.cursor()
        print(f"\n🔍 Executing SQL Query on {database_path} ...")
        start_time = time.time()
        cursor.execute(sql_query)
        results = cursor.fetchall()
        end_time = time.time()
        execution_time = end_time - start_time

        time_detailsSM1.append({
            "Database": database_path,
            "Query": sql_query,
            "Execution Time": execution_time
        })

        if scaling_factor != 1:
            query_upper = sql_query.upper()
            if ("SUM(" in query_upper or "COUNT(" in query_upper) and "AVG(" not in query_upper:
                if len(results) == 1 and isinstance(results[0][0], (int, float)):
                    results = [[results[0][0] * scaling_factor]]
                else:
                    scaled_results = []
                    for row in results:
                        new_row = [val * scaling_factor if isinstance(val, (int, float)) else val for val in row]
                        scaled_results.append(tuple(new_row))
                    results = scaled_results

        conn.close()
        return results, execution_time
    except Exception as e:
        print(f"⚠️ Error: {e}")
        return None, 0

def aggregate_results(results):
    try:
        return sum(row[0] for row in results if isinstance(row[0], (int, float)))
    except Exception as e:
        print(f"Error aggregating results: {e}")
        return None

def compute_accuracy(gold_value, sample_value):
    try:
        error = abs(gold_value - sample_value)
        percent_error = (error / gold_value * 100) if gold_value != 0 else 0
        return error, percent_error
    except Exception as e:
        print(f"Error computing accuracy: {e}")
        return None, None

def  export_to_csv():
    with open("experiment_results.csv", "w", newline="") as file:
        writer = csv.writer(file)
        headers = ["Query", "Database", "Aggregated Result", "Execution Time (s)", "Absolute Error", "Percentage Error"]
        writer.writerow(headers)
        for row in execution_summary:
            writer.writerow(row)

def run_experiment(sql_query, large_db_path, sample_db_paths_with_size):
    gold_results, gold_time = execute_sql(large_db_path, sql_query, scaling_factor=1)
    gold_aggregated = aggregate_results(gold_results) if gold_results else None

    print("\n--- Gold Standard (Large Database) ---")
    print(f"Execution Time: {gold_time:.4f} sec")
    print(f"Aggregated Result: {gold_aggregated if gold_aggregated is not None else 'Non-numeric'}")

    for db_name, db_size in sample_db_paths_with_size.items():
        scaling_factor = db_size / 1073741824
        sample_results, sample_time = execute_sql(db_name, sql_query, scaling_factor=(1 / scaling_factor))
        sample_aggregated = aggregate_results(sample_results) if sample_results else None

        if gold_aggregated is not None and sample_aggregated is not None:
            abs_error, percent_error = compute_accuracy(gold_aggregated, sample_aggregated)
        else:
            abs_error, percent_error = None, None

        print(f"\n--- Sample Database: {db_name} ---")
        print(f"Size: {db_size / (1024*1024):.0f} MB")
        print(f"Scaling Factor: {(1 / scaling_factor):.2f}")
        print(f"Execution Time: {sample_time:.4f} sec")
        print(f"Aggregated Sample Result: {sample_aggregated}")
        print(f"Absolute Error: {abs_error}")
        print(f"Percentage Error: {percent_error:.2f}%" if percent_error is not None else "Error not computable")

        execution_summary.append([
            sql_query,
            db_name,
            sample_aggregated,
            f"{sample_time:.4f}",
            f"{abs_error:.4f}" if abs_error is not None else None,
            f"{percent_error:.2f}%" if percent_error is not None else None
        ])

def main():
    large_db_path = "large_database.db"
    sample_db_paths_with_size = {
        '50mb_database.db': 52428800,
        '100mb_database.db': 104857600,
        '150mb_database.db': 157286400,
        '200mb_database.db': 209715200,
        '250mb_database.db': 262144000
    }

    while True:
        user_command = input("Enter your query (or type 'exit' to quit): ")
        if user_command.lower() == "exit":
            break

        sql_query = natural_language_to_sql(user_command)
        print(f"\n📝 Generated SQL Query:\n{sql_query}")
        run_experiment(sql_query, large_db_path, sample_db_paths_with_size)

        print("\n--- Execution Summary ---")
        for row in execution_summary:
            print(row)

        time_detailsSM1.clear()

    export_to_csv()  # Only write to CSV after user exits

if __name__ == "__main__":
    main()


Enter your query (or type 'exit' to quit):  Total number of customers



🔍 Model Raw Output:
 SELECT COUNT(*) FROM customers;

📝 Generated SQL Query:
SELECT COUNT(*) FROM customers;

🔍 Executing SQL Query on large_database.db ...

--- Gold Standard (Large Database) ---
Execution Time: 27.5462 sec
Aggregated Result: 3090885

🔍 Executing SQL Query on 50mb_database.db ...
⚠️ Error: no such table: customers

--- Sample Database: 50mb_database.db ---
Size: 50 MB
Scaling Factor: 20.48
Execution Time: 0.0000 sec
Aggregated Sample Result: None
Absolute Error: None
Error not computable

🔍 Executing SQL Query on 100mb_database.db ...
⚠️ Error: no such table: customers

--- Sample Database: 100mb_database.db ---
Size: 100 MB
Scaling Factor: 10.24
Execution Time: 0.0000 sec
Aggregated Sample Result: None
Absolute Error: None
Error not computable

🔍 Executing SQL Query on 150mb_database.db ...
⚠️ Error: no such table: customers

--- Sample Database: 150mb_database.db ---
Size: 150 MB
Scaling Factor: 6.83
Execution Time: 0.0000 sec
Aggregated Sample Result: None
Absolut

Enter your query (or type 'exit' to quit):  exit


New code

In [9]:
import requests
import json
import sqlite3
import time
import csv
import os
from tqdm import tqdm

# API setup
url = "http://localhost:8000/v1/chat/completions"
headers = {"Content-Type": "application/json"}

execution_summary = []
nl_sql_log = []

def ask_model(prompt):
    data = {
        "model": "defog/llama-3-sqlcoder-8b",
        "temperature": 0.0001,
        "messages": [{"role": "user", "content": prompt}]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    if response.status_code == 200:
        model_output = response.json()["choices"][0]["message"]["content"]
        print("\n🔍 Model Raw Output:\n", model_output)
        return model_output.strip()
    else:
        raise Exception(f"API request failed: {response.status_code}")

def natural_language_to_sql(user_command):
    prompt = f"""
### Task:
Convert the following natural language command into a correctly formatted SQLite SQL query.

### Notes:
- Use SQLite-compatible functions only.
- Avoid EXTRACT, AGE, TO_DATE, DATE_PART, etc.
- Use julianday for date math if needed.
- Return ONLY the SQL query (no markdown, no explanation).

Table: customers
Columns:
  id, name, email, address, phone_number, job, company, date_of_birth,
  ssn, credit_card_number, credit_card_expire, credit_card_provider,
  bank_country, currency_code, amount, transaction_date

### User Input:
{user_command}

### SQL Output:
"""
    sql = ask_model(prompt)
    nl_sql_log.append([user_command, sql])
    if sql.strip().upper().startswith("SELECT") and sql.strip().endswith(";"):
        return sql.strip()
    else:
        print("⚠️ Warning: Invalid SQL returned. Using fallback.")
        return "SELECT * FROM customers LIMIT 5;"

def execute_sql(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    start = time.time()
    cur.execute(sql_query)
    result = cur.fetchall()
    end = time.time()
    conn.close()
    return result, end - start

def aggregate_results(results):
    try:
        return sum(row[0] for row in results if isinstance(row[0], (int, float)))
    except:
        return None

def compute_relative_error(gold, sample):
    try:
        return abs(sample - gold) / gold * 100 if gold != 0 else None
    except:
        return None

def compute_speed_overhead(gold_time, sample_time):
    try:
        return ((sample_time / gold_time) - 1) * 100 if gold_time != 0 else None
    except:
        return None

def export_summary_results():
    with open("experiment_results.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Raw Sample Result", "Scaled Result", "Execution Time (s)", "Relative Error (%)", "Speed Overhead (%)"])
        writer.writerows(execution_summary)

def export_query_txt():
    with open("queries_log.txt", "w") as f:
        for nl, sql in nl_sql_log:
            f.write(f"Natural Language: {nl}\nSQL Query: {sql}\n\n")

def export_error_overhead():
    with open("errors_overhead.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Relative Error (%)", "Speed Overhead (%)"])
        for row in execution_summary:
            writer.writerow([row[0], row[1], row[5], row[6]])

def run_experiment(sql_query, large_db_path, sample_db_paths_with_size):
    print("\n🔍 Executing on full database...")
    gold_results, gold_time = execute_sql(large_db_path, sql_query)
    gold_aggregated = aggregate_results(gold_results)

    print(f"✅ Full DB Result: {gold_aggregated} | Time: {gold_time:.4f}s")

    for db_path, size_bytes in tqdm(sample_db_paths_with_size.items(), desc="🔁 Sample DBs"):
        scaling_factor = size_bytes / os.path.getsize(large_db_path)
        inverse_scaling = 1 / scaling_factor

        try:
            sample_results, sample_time = execute_sql(db_path, sql_query)
            raw_result = aggregate_results(sample_results)

            scaled_result = raw_result * inverse_scaling if raw_result is not None else None
            rel_error = compute_relative_error(gold_aggregated, scaled_result)
            speed_overhead = compute_speed_overhead(gold_time, sample_time)

            print(f"\n📁 {db_path}")
            print(f"Raw: {raw_result}, Scaled: {scaled_result:.2f} | Time: {sample_time:.4f}s")
            print(f"Relative Error: {rel_error:.2f}%" if rel_error is not None else "N/A")
            print(f"Speed Overhead: {speed_overhead:.2f}%" if speed_overhead is not None else "N/A")

            execution_summary.append([
                sql_query,
                db_path,
                raw_result,
                scaled_result,
                f"{sample_time:.4f}",
                f"{rel_error:.2f}" if rel_error is not None else None,
                f"{speed_overhead:.2f}" if speed_overhead is not None else None
            ])

        except Exception as e:
            print(f"❌ Failed on {db_path}: {e}")
            execution_summary.append([sql_query, db_path, None, None, None, None, None])

def main():
    large_db_path = "large_database.db"
    sample_db_paths_with_size = {
        "50mb_sample.db": 52428800,
        "100mb_sample.db": 104857600,
        "150mb_sample.db": 157286400,
        "200mb_sample.db": 209715200,
        "250mb_sample.db": 262144000
    }

    while True:
        user_input = input("\n🧠 Enter your query (or type 'exit'): ")
        if user_input.lower() == "exit":
            break

        sql_query = natural_language_to_sql(user_input)
        print(f"\n📝 SQL Query:\n{sql_query}")
        run_experiment(sql_query, large_db_path, sample_db_paths_with_size)

    export_summary_results()
    export_query_txt()
    export_error_overhead()

    print("\n✅ Saved:")
    print(" - experiment_results.csv (full details)")
    print(" - queries_log.txt (natural input + SQL)")
    print(" - errors_overhead.csv (relative error + speed overhead only)")

if __name__ == "__main__":
    main()



🧠 Enter your query (or type 'exit'):   What is the total transaction amount recorded in the database?



🔍 Model Raw Output:
 SELECT SUM(c.amount) AS total_transaction_amount FROM customers c;

📝 SQL Query:
SELECT SUM(c.amount) AS total_transaction_amount FROM customers c;

🔍 Executing on full database...
✅ Full DB Result: 15601762838.060753 | Time: 5.3182s


🔁 Sample DBs:  20%|██        | 1/5 [00:00<00:01,  2.25it/s]


📁 50mb_sample.db
Raw: 761927046.4099996, Scaled: 15605694523.69 | Time: 0.4300s
Relative Error: 0.03%
Speed Overhead: -91.92%


🔁 Sample DBs:  40%|████      | 2/5 [00:01<00:01,  1.68it/s]

...


📁 200mb_sample.db
Raw: 150803, Scaled: 772182.05 | Time: 0.9176s
Relative Error: 0.28%
Speed Overhead: -80.16%


🔁 Sample DBs: 100%|██████████| 5/5 [00:03<00:00,  1.29it/s]



📁 250mb_sample.db
Raw: 188307, Scaled: 771376.09 | Time: 1.1742s
Relative Error: 0.17%
Speed Overhead: -74.62%



🧠 Enter your query (or type 'exit'):  exit



✅ Saved:
 - experiment_results.csv (full details)
 - queries_log.txt (natural input + SQL)
 - errors_overhead.csv (relative error + speed overhead only)


In [10]:
import csv

input_file = "errors_overhead.csv"
output_file = "errors_overhead_cleaned.csv"

with open(input_file, "r", newline="") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    headers = next(reader)
    writer.writerow(headers)

    for row in reader:
        query, db_name, rel_error, speed_overhead = row

        # Clean speed overhead
        if speed_overhead and speed_overhead != "None":
            try:
                speed_overhead_value = abs(float(speed_overhead.replace("%", "")))
                speed_overhead = f"{speed_overhead_value:.2f}"
            except:
                pass

        writer.writerow([query, db_name, rel_error, speed_overhead])
